In [0]:
%run ../../../utils/pipeline_utils

In [0]:
# Databricks notebook source
# =============================================================================
# common/prime_dim_line_item.py
# Primes assist_dev.common.dim_line_item
#
# Source tables (Silver):
#   silver_aasbs_line_item          → CLIN master record
#   silver_aasbs_line_item_accepted → accepted instance (budget_fy, exercise_this_yn)
#   silver_aasbs_lu_line_item_type  → line_item_type_desc decoded
#   silver_aasbs_lu_contract_type   → contract_type_desc decoded at CLIN level
#
# Grain       : One row per line_item_accepted.id
#               (a CLIN can have multiple accepted instances across mods;
#                on prime, we take one row per distinct line_item_id with the
#                latest accepted record)
# SCD Type    : 2 — eff_start_dt=NOW, eff_end_dt=NULL, is_current_flag=TRUE
# Idempotent  : YES — TRUNCATE then INSERT
# Dependencies: dim_award (for award_sk)
#
# Source column notes:
#   line_item.id                    → line_item_id (natural key)
#   line_item.li_tracking_num       → li_tracking_num (stable across mods)
#   line_item.line_item_type_cd     → line_item_type_cd
#   line_item.line_item_num         → clin_num (e.g. '0001', '0002AA')
#   line_item.line_item_start_dt    → li_pop_start_dt
#   line_item.line_item_end_dt      → li_pop_end_dt
#   line_item.line_item_total_amt   → obligated_ceiling_amt
#   line_item.parent_award_mod_id   → FK to award_mod → award_id → award_sk
#   line_item_accepted.id           → line_item_accepted_id
#   line_item_accepted.budget_fy    → budget_fy
#   line_item_accepted.exercise_this_yn → exercise_this_yn (Y/N → BOOLEAN)
#
#   slin_num: ASSIST stores CLINs with a multi-part line_item_num. Numbers
#             with length > 4 chars are SLINs (e.g. '0001AA' → clin=0001, slin=AA).
#             We split on character position 4 for simplicity.
#   psc_cd:   Not on line_item in source DDL. Stored on li_deliverable.
#             Pull from silver_aasbs_li_deliverable if present; NULL otherwise.
#   unit_of_measure_cd: Not on line_item in source DDL. NULL on prime.
#   contract_type_cd: Not on line_item directly. Stored on li_deliverable or
#                     award_mod. NULL on prime; enriched via delta refresh.
#   severability_type_cd: Not in line_item source DDL. NULL on prime.
#   bona_fide_need_fy: Not in line_item source DDL. Derived from LOA fiscal year.
# =============================================================================

# COMMAND ----------
# MAGIC %run ../utils/pipeline_utils
# COMMAND ----------
dbutils.widgets.text("run_id",   "", "Pipeline Run ID")
dbutils.widgets.text("job_name", "dp1_prime_full", "Job Name")

RUN_ID   = dbutils.widgets.get("run_id")   or "manual-" + get_spark_app_id()
JOB_NAME = dbutils.widgets.get("job_name")

TARGET   = gold("common", "dim_line_item")
TASK     = "prime_dim_line_item"

S_LI     = silver("aasbs", "line_item")
S_LIA    = silver("aasbs", "line_item_accepted")
S_LIT    = silver("aasbs", "lu_line_item_type")
S_CT     = silver("aasbs", "lu_contract_type")
S_MOD    = silver("aasbs", "award_mod")
G_AWARD  = gold("common", "dim_award")

print(f"[{TASK}] target={TARGET}")

# COMMAND ----------
start_ts = audit_start(spark, RUN_ID, JOB_NAME, TASK, TARGET,
                        source_schema="aasbs", source_table="line_item")

# COMMAND ----------
try:
    truncate_gold(spark, TARGET)

    # line_item_sk is GENERATED ALWAYS AS IDENTITY — excluded from INSERT col list.
    #
    # Strategy for grain selection on prime:
    #   Use the latest line_item_accepted record per line_item_id
    #   (MAX(line_item_accepted.id) per line_item_id, since IDs are monotonically
    #    increasing with each award mod exercise).
    #   Delta refresh will insert new SCD2 versions when line_items change.

    spark.sql(f"""
        INSERT INTO {TARGET}
        (
            line_item_id, line_item_accepted_id, li_tracking_num,
            award_sk,
            clin_num, slin_num,
            line_item_type_cd, line_item_type_desc,
            psc_cd, unit_of_measure_cd,
            contract_type_cd, severability_type_cd,
            li_pop_start_dt, li_pop_end_dt,
            obligated_ceiling_amt,
            exercise_this_yn, bona_fide_need_fy, budget_fy,
            eff_start_dt, eff_end_dt, is_current_flag,
            _gold_created_at, _gold_updated_at, _source_batch_id
        )
        WITH latest_accepted AS (
            -- One accepted record per line_item_id: take latest by accepted.id
            SELECT
                line_item_id,
                MAX(id)         AS line_item_accepted_id
            FROM {S_LIA}
            WHERE NVL(is_deleted,FALSE) = FALSE
            GROUP BY line_item_id
        ),
        accepted_detail AS (
            SELECT
                la.line_item_id,
                line_item_accepted_id,
                --la.id                               AS line_item_accepted_id,
                lia.budget_fy,
                lia.exercise_this_yn,
                lia.award_mod_id                    AS accepted_award_mod_id
            FROM latest_accepted la
            JOIN {S_LIA} lia
                ON lia.id = la.line_item_accepted_id
                AND NVL(lia.is_deleted,FALSE) = FALSE
        ),
        li_core AS (
            SELECT
                li.id                               AS line_item_id,
                li.li_tracking_num,
                li.line_item_type_cd,
                -- clin_num: the full line_item_num (e.g. '0001', '0001AA', '0001AB')
                li.line_item_num                    AS clin_num,
                -- slin_num: the portion after position 4 (non-numeric suffix)
                -- '0001AA' → clin='0001', slin='AA'
                CASE
                    WHEN LENGTH(TRIM(li.line_item_num)) > 4
                    THEN SUBSTRING(TRIM(li.line_item_num), 5)
                    ELSE NULL
                END                                 AS slin_num,
                li.line_item_start_dt               AS li_pop_start_dt,
                li.line_item_end_dt                 AS li_pop_end_dt,
                li.line_item_total_amt              AS obligated_ceiling_amt,
                li.line_item_fy                     AS li_fy,
                -- Parent award mod: used to resolve award_sk
                li.parent_award_mod_id
            FROM {S_LI} li
            WHERE NVL(li.is_deleted,FALSE) = FALSE
        ),
        with_accepted AS (
            SELECT
                lc.*,
                COALESCE(ad.line_item_accepted_id, NULL)  AS line_item_accepted_id,
                COALESCE(ad.budget_fy, lc.li_fy)          AS budget_fy,
                -- exercise_this_yn: 'Y'/'N' character → BOOLEAN
                CASE WHEN ad.exercise_this_yn = 'Y' THEN TRUE
                     WHEN ad.exercise_this_yn = 'N' THEN FALSE
                     ELSE NULL
                END                                        AS exercise_this_yn
            FROM li_core lc
            LEFT JOIN accepted_detail ad
                ON ad.line_item_id = lc.line_item_id
        ),
        with_award_sk AS (
            -- Resolve award_sk via parent_award_mod_id → award_mod.award_id → dim_award
            SELECT
                wa.*,
                aw.award_sk
            FROM with_accepted wa
            LEFT JOIN {S_MOD} m
                ON m.id = wa.parent_award_mod_id
                AND NVL(m.is_deleted,FALSE) = FALSE
            LEFT JOIN {G_AWARD} aw
                ON aw.award_id = m.award_id
                AND NVL(aw.is_current_flag,FALSE) = TRUE
        ),
        with_decoded AS (
            SELECT
                ws.*,
                COALESCE(lit.description, ws.line_item_type_cd) AS line_item_type_desc
            FROM with_award_sk ws
            LEFT JOIN {S_LIT} lit
                ON ws.line_item_type_cd = lit.cd
                AND NVL(lit.is_deleted,FALSE) = FALSE
        )
        SELECT
            line_item_id,
            line_item_accepted_id,
            li_tracking_num,
            award_sk,
            clin_num,
            slin_num,
            line_item_type_cd,
            line_item_type_desc,
            -- psc_cd: from li_deliverable (not joined here; see note above)
            CAST(NULL AS STRING)                    AS psc_cd,
            -- unit_of_measure_cd: not in source line_item DDL
            CAST(NULL AS STRING)                    AS unit_of_measure_cd,
            -- contract_type_cd: from li_deliverable or award_mod
            CAST(NULL AS STRING)                    AS contract_type_cd,
            -- severability_type_cd: not in source line_item DDL
            CAST(NULL AS STRING)                    AS severability_type_cd,
            li_pop_start_dt,
            li_pop_end_dt,
            obligated_ceiling_amt,
            exercise_this_yn,
            -- bona_fide_need_fy: derived from line_item_fy (fiscal year of CLIN)
            li_fy                                   AS bona_fide_need_fy,
            budget_fy,
            -- SCD2 initial values
            CURRENT_TIMESTAMP()                     AS eff_start_dt,
            CAST(NULL AS TIMESTAMP)                 AS eff_end_dt,
            TRUE                                    AS is_current_flag,
            CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), '{RUN_ID}'
        FROM with_decoded
    """)

    n = row_count(spark, TARGET)
    no_award = spark.sql(
        f"SELECT COUNT(*) AS n FROM {TARGET} WHERE award_sk IS NULL"
    ).collect()[0]["n"]
    no_clin = spark.sql(
        f"SELECT COUNT(*) AS n FROM {TARGET} WHERE clin_num IS NULL"
    ).collect()[0]["n"]
    print(f"  [OK] Inserted {n:,} CLIN rows")
    print(f"       {no_award:,} with unresolved award_sk | {no_clin:,} with NULL clin_num")

    audit_success(spark, RUN_ID, TARGET, n, n, start_ts)

except Exception as e:
    audit_fail(spark, RUN_ID, TARGET, str(e), traceback.format_exc(), start_ts)
    raise

# COMMAND ----------
#dbutils.notebook.exit("SUCCESS")
